<a href="https://colab.research.google.com/github/ladparag100/AI-Projects/blob/main/Multi_Agent_Travel_Planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# **Travel Planning Multi-Agent system**

An intelligent system that coordinates multiple specialized agents to search flights, hotels, plan itineraries, and provide personalized travel recommendations.

# **Step 1 - Start with the process to automate**

*   Search hotels and flights based on user preferences
*   Plan complete travel itineraries
*   Answer travel-related questions with deep research
*   Handle multiple queries in parallel
*   Present unified, well-formatted travel recommendation

# **Step 2 & 3 - Map out your workflow as discrete steps & Identify what each steps need to do**

*   **Search Agent:** Handles hotel and flight search based on user preferences.
*   **Itinerary Planner Agent:** Utilizes search results and research to create detailed travel plans.

## Install LangGraph

In [ ]:
# from openai import OpenAI
# client = OpenAI()
# response = client.responses.create(
#     model="gpt-5.5",
#     input="Write a short bedtime story about a unicorn."
# )
# print(response)

In [ ]:
!pip install -U -q langgraph langchain

## Chat Model

In [ ]:
# Install Required Libraries
!pip install -U -q langchain-openai

In [ ]:
# Import API Keys
import getpass
import os
from google.colab import userdata

try:
    api_key_from_secrets = userdata.get("OPENAI_API_KEY")
    os.environ["OPENAI_API_KEY"] = api_key_from_secrets
except Exception:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [ ]:
# Chat Model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4",
    temperature=0.2,
)

### Test Chat Model

In [ ]:
test_msg = llm.invoke("What is Machine Learning? Explain with the help of an example.")

In [ ]:
test_msg

In [ ]:
# Print in Readable Format
print(test_msg.content)

## **Itinerary Agent or Scout**

In [ ]:
# Install Required Libraries
!pip install -q deepagents tavily-python langchain-tavily

### Web Search Tool

In [ ]:
# Necessary Imports
from typing import Literal
from tavily import TavilyClient

In [ ]:
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent

In [ ]:
# Import API Keys
try:
    api_key_from_secrets = userdata.get("TAVILY_API_KEY")
    os.environ["TAVILY_API_KEY"] = api_key_from_secrets
except Exception:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

In [ ]:
# Create Internet Search via TavilySearch
from langchain_tavily import TavilySearch

internet_search = TavilySearch(
    max_results=5,
    topic="general",
    include_images=True,
    include_image_descriptions=True,
    search_depth="advanced",
)

In [ ]:
# Test internet_search tavily
internet_search.invoke({"query": "Top Travel attractions in Vienna"})

### **Deep Research Agent**

In [ ]:
# Define Model for Deep Research Agent
model = init_chat_model(model="gpt-4",
                        model_provider="openai",
                        temperature=0.2)

In [ ]:
# Itinerary Agent (Deep Research) Prompt
from deepagents import create_deep_agent

research_instructions = """You are a professional travel itinerary planning agent specializing exclusively in trip research and itinerary design.

SCOPE AND BEHAVIOR RULES
- Respond ONLY to travel-related requests, including destinations, itineraries, activities, transportation, accommodations, budgeting, and travel logistics.
- If a request is unrelated to travel (e.g., math, general knowledge, coding, weather outside trip context), politely decline and redirect the user to a travel-planning request.
- Do NOT answer hypothetical or fictional travel questions unless explicitly stated by the user.

RESEARCH AND REASONING PROCESS (ReAct)
You MUST follow this process internally:
1. THOUGHT: Analyze the user's travel goals, constraints, preferences, and missing information.
2. ACTION: Use TavilySearch to retrieve current, authoritative travel data (attractions, hours, pricing, transportation options, seasonal considerations).
3. OBSERVATION: Evaluate and synthesize search results; resolve conflicts or note uncertainty when needed.
4. RESPONSE: Produce a complete, user-ready itinerary.

TOOL USAGE
- TavilySearch is the primary tool for researching up-to-date travel information.
- Prefer official tourism boards, transportation providers, reputable travel guides, and recent reviews.
- Do not fabricate details if information is unavailable; explicitly state assumptions or gaps.

OUTPUT REQUIREMENTS
All itineraries MUST include:
- A clear day-by-day structure (Day 1, Day 2, etc.)
- Specific activity timing (morning / afternoon / evening, with approximate hours)
- Exact locations or neighborhoods
- Transportation methods between stops (walking, public transit, taxi, flight, etc.)
- Estimated costs (ranges are acceptable)
- Practical tips (tickets, reservations, safety, local customs)

FORMATTING GUIDELINES
- Use clear headings and bullet points
- Optimize for readability and execution during travel
- Be concise but thorough; avoid filler or generic advice

QUALITY BAR
- Prioritize realism, efficiency, and traveler experience
- Tailor recommendations to trip duration, pace, and traveler type when information is available
- If critical details are missing, ask targeted clarification questions before finalizing the itinerary
"""

In [ ]:
# Itinerary Agent (deep research agent)
itinerary_research_agent = create_deep_agent(
    model=model,
    system_prompt=research_instructions,
    tools=[internet_search],
)

In [ ]:
# Test Deep Research Agent (Itinerary Agent)
result = itinerary_research_agent.invoke({"messages": [{"role": "user",
                                                        "content": "Plan a 5 day travel itinerary for a food & historical sites lover in Tanzania"}]})

print(result)

In [ ]:
# Print the agent's response
print(result["messages"][-1].content)

In [ ]:
from IPython.display import Image, display

display(Image(itinerary_research_agent.get_graph().draw_mermaid_png()))

### **`itinerary_research_agent` as a TOOL (Sub Agent)**

In [ ]:
# Defining itinerary_research_agent as a Tool
from langchain.tools import tool

@tool("itinerary_research_agent", description="plans travel itinerary")
def call_itinerary_research_agent(query: str):
    result = itinerary_research_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

# **Travel Scout Agent = LLM + Prompt + Tools (Deep Research Tool & Tavily)**

### **Travel Scout Tools**

In [ ]:
tools=[internet_search, call_itinerary_research_agent]

### **Travel Scout Prompt**

In [ ]:
# Travel Scout Prompt
travel_scout_instructions = """You are a General Travel Scout specializing in both high-level travel information, guidance and itinerary planning for multi-day trips.
Your role is to answer both general travel questions and questions that require detailed itinerary planning.

SCOPE AND BEHAVIOR RULES:
- Respond ONLY to general travel-related queries, such as:
  - Weather and climate of destinations
  - Best cities or regions to visit by season or interest
  - What to pack or wear (clothing, gear, cultural norms)
  - Safety, visas, currency, local customs, and basic logistics
  - High-level comparisons between destinations
- To create day-by-day itineraries if asked use **itinerary_research_agent**.
- Do NOT search or recommend specific flights or hotels.
- Politely decline and redirect if the request is unrelated to travel.

TOOL SELECTION RULES (CRITICAL)
USE **internet_search** for:
- General travel questions
- High-level guidance and quick factual lookups
- Topics that do NOT require structured planning or multi-day sequencing

Examples:
- Weather or climate at a destination
- Best time to visit a country or city
- Visa requirements or entry rules
- Local culture, etiquette, and customs
- Safety considerations and travel advisories
- Currency, language, SIM cards, transportation basics
- Packing tips and clothing advice
- High-level destination comparisons
- Popular attractions (without scheduling)

USE **itinerary_research_agent** for:
- Any request that requires structured planning or sequencing
- Multi-day or day-by-day travel plans
- Deep destination research across multiple locations
- Experience-based optimization (pace, routes, themes)

PROCESS (MANDATORY):
1. Identify the intent and depth of the travel question.
2. Select the correct tool based on Tool Selection Rules.
3. Execute the tool.
4. Synthesize results into a clear, concise, traveler-friendly response.
5. State assumptions, seasonal variations, or uncertainty if applicable.

OUTPUT REQUIREMENTS:
- Provide a direct, practical answer optimized for quick decision-making.
- Avoid deep research, long narratives, or detailed schedules.
- Include actionable tips when helpful (e.g., 'best months,' 'what to avoid,' 'what to pack').

SOURCE CITATION (REQUIRED):
- Always include a short 'Sources' section at the end.
- Cite 2–4 reputable sources used via TavilySearch.

FORMATTING GUIDELINES:
- Use clear headings and bullet points
- Keep responses concise, informative, and easy to scan
- Avoid filler, marketing language, or speculative advice
"""

In [ ]:
model = llm

In [ ]:
# TRAVEL SCOUT (INTERNET SEARCH + DEEP RESEARCH)/REACT
from langchain.agents import create_agent

travel_scout = create_agent(
    model=model,
    system_prompt=travel_scout_instructions,
    tools=tools
)

### Test Travel Scout

In [ ]:
travel_scout_result = travel_scout.invoke(
    {"messages": [{"role": "user",
                   "content": "What is the best season to travel to Jodhpur?"}]}
)
print(travel_scout_result["messages"][-1].content)

In [ ]:
travel_scout_result = travel_scout.invoke(
    {"messages": [{"role": "user",
                   "content": "Plan 2 day itinerary to Jodhpur from Boston for a history & culture lover, also want to try special cuisine to this city"}]}
)
print(travel_scout_result["messages"][-1].content)

### **Streaming**

In [ ]:
# Stream events
query = "What is the best season to travel to Jodhpur?"

for chunk in travel_scout.stream(
    {"messages": [{"role": "user",
                   "content": query}]}
):
    print(chunk)

In [ ]:
for event in travel_scout.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",):
    print(event)
    event["messages"][-1].pretty_print()

In [ ]:
# Full state of the graph after each node finishes its work
query = "Plan 2 day itinerary for Agra travel for a history & culture lover, also want to try special cuisine to this city"
for event in travel_scout.stream(
    {"messages": [{"role": "user",
                   "content": query}]},
    stream_mode="values",):
    print(event)
    event["messages"][-1].pretty_print()